# Lecture 10C.2 — openSMILE eGeMAPS: Standardized Acoustic Features


## Goal

Use openSMILE (via the Python `opensmile` package) to extract:

- eGeMAPS (a compact, widely used feature set)
- low-level descriptors + functionals

This gives you a **high-coverage baseline** for emotion, health, and paralinguistics tasks.


## 1) Setup


In [1]:
import os, json, re, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional audio playback
try:
    from IPython.display import Audio, display
    HAS_IPY_AUDIO = True
except Exception:
    HAS_IPY_AUDIO = False

# ---------------- Project paths (shared manifest workflow) ----------------
PROJECT_ROOT = Path.cwd() / "EE519_L10C_Project"
REC_DIR = PROJECT_ROOT / "recordings"
FIG_DIR = PROJECT_ROOT / "figures"
RES_DIR = PROJECT_ROOT / "results"
FEAT_DIR = PROJECT_ROOT / "features"
MANIFEST_PATH = PROJECT_ROOT / "manifest.json"

for d in [REC_DIR, FIG_DIR, RES_DIR, FEAT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def load_manifest(path=MANIFEST_PATH):
    if path.exists():
        return json.loads(path.read_text())
    return {"course":"EE519","module":"Lecture10C","created_utc":None,"clips":[],"splits":{}}

def save_manifest(manifest, path=MANIFEST_PATH):
    if manifest.get("created_utc") is None:
        manifest["created_utc"] = str(np.datetime64("now"))
    path.write_text(json.dumps(manifest, indent=2))
    print("Saved manifest:", path)

def save_fig(fig, name, dpi=150):
    out = FIG_DIR / name
    fig.savefig(out, dpi=dpi, bbox_inches="tight")
    print("Saved:", out)
    return out

import wave
def read_wav(path: Path):
    with wave.open(str(path), "rb") as wf:
        fs = wf.getframerate()
        n = wf.getnframes()
        x = np.frombuffer(wf.readframes(n), dtype=np.int16).astype(np.float32) / 32768.0
    return fs, x

def peak_normalize(x, target=0.98):
    m = np.max(np.abs(x)) + 1e-12
    return (x / m) * target

def play_audio(x, fs, label="audio"):
    if not HAS_IPY_AUDIO:
        print("(Audio playback not available)", label)
        return
    display(Audio(x, rate=fs))

def list_clips(manifest):
    for i,c in enumerate(manifest.get("clips", [])):
        print(f"[{i}] {c.get('label',''):14s}  {c.get('filename','')}  fs={c.get('fs','?')}  notes={c.get('notes','')}")

def get_segments(clip):
    # We reuse analysis_segments from earlier threads if present
    return clip.get("selections", {}).get("analysis_segments", {})

manifest = load_manifest()
print("Project root:", PROJECT_ROOT)
print("Clips:", len(manifest.get('clips', [])))
list_clips(manifest)


Project root: /Users/kadiris1/Desktop/Couses-USC/Experiential learning/L10/EE519_Lecture10C_Notebooks_Release_v1/EE519_L10C_Project
Clips: 4
[0] vowel_a         student10B_vowel_a.wav  fs=16000  notes=steady /a/
[1] vowel_i         student10B_vowel_i.wav  fs=16000  notes=steady /i/
[2] fricative_s     student10B_fricative_s.wav  fs=16000  notes=steady /s/
[3] sentence        student10B_sentence.wav  fs=16000  notes=short sentence


## 2) Install / import opensmile (with fallback)

If `opensmile` isn’t installed, you can still read this notebook and run later.


In [2]:
#pip install --upgrade pip setuptools wheel

In [12]:
#pip install opensmile

In [9]:
#brew install opensmile

In [10]:
#pip install --upgrade pip setuptools wheel

In [14]:
#python3 -m pip install --upgrade pip setuptools wheel

In [11]:
HAS_SMILE = True
try:
    import opensmile
except Exception as e:
    HAS_SMILE = False
    print("opensmile not available:", repr(e))
    print("Install: pip install opensmile")


opensmile not available: ModuleNotFoundError("No module named 'pkg_resources'")
Install: pip install opensmile


## 3) Extract eGeMAPS per segment

We will:
- read each segment from manifest
- run openSMILE
- save one row per segment


In [ ]:
manifest = load_manifest()

# Build segment list
items = []
for ci, clip in enumerate(manifest.get("clips", [])):
    for seg_name, sel in get_segments(clip).items():
        items.append((ci, seg_name, sel))

print("Segments:", len(items))

if not HAS_SMILE:
    print("Skipping extraction because opensmile is missing.")
else:
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )

    feat_rows = []
    for ci, seg_name, sel in items:
        clip = manifest["clips"][ci]
        wav_path = REC_DIR / clip["filename"]
        fs, x = read_wav(wav_path); x = peak_normalize(x)
        xseg = x[int(sel["s0"]):int(sel["s1"])]
        if len(xseg) < int(0.1*fs):
            # too short for stable statistics
            continue
        # opensmile can process arrays directly
        df = smile.process_signal(xseg, fs)
        # flatten to dict
        r = df.iloc[0].to_dict()
        r.update({"clip_idx":ci, "segment":seg_name, "label":clip.get("label",""), "filename":clip.get("filename","")})
        feat_rows.append(r)

    smile_df = pd.DataFrame(feat_rows)
    display(smile_df.head())

    out_csv = FEAT_DIR / "opensmile_eGeMAPS_segments.csv"
    smile_df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)


## 4) Quick exploration: which features separate vowel vs fricative?

We’ll:
- join with the segment split table (if created in 10C.0)
- compute simple correlations / separations


In [ ]:
split_csv = RES_DIR / "L10C_segment_split.csv"
if not split_csv.exists():
    print("No split table found. Run 10C.0 first.")
elif not ('smile_df' in globals()):
    print("No openSMILE table in memory. Run extraction above (requires opensmile).")
else:
    split_df = pd.read_csv(split_csv)
    # merge on clip_idx+segment
    merged = split_df.merge(smile_df, on=["clip_idx","segment","label","filename"], how="inner")
    print("Merged rows:", len(merged))

    # pick a few interpretable eGeMAPS examples (names may vary slightly)
    cols = [c for c in merged.columns if any(k in c.lower() for k in ["f0", "jitter", "shimmer", "h1", "alpha", "slope", "loudness", "mfcc"])]
    cols = cols[:12]
    display(merged[["group","split"] + cols].head())

    # simple group means
    means = merged.groupby("group")[cols].mean()
    display(means)


## Reflection questions

1) Why are standardized sets like eGeMAPS useful for benchmarking?  
2) Which openSMILE features would you expect to correlate with “loud vs soft”?  
3) Why do “functionals” (mean/std/percentiles) matter for variable-length segments?


## What’s next
- **10C.3** Self-supervised embeddings (wav2vec2/HuBERT): feature learning instead of hand-crafting.
